In [ ]:
# Cell 1: Kết nối
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

print("Connected!")

Connected!


In [17]:
df_video    = pd.read_sql("SELECT * FROM public.dim_video", engine)
df_channel  = pd.read_sql("SELECT * FROM public.dim_channel", engine)
df_topic    = pd.read_sql("SELECT * FROM public.dim_topic", engine)
df_time     = pd.read_sql("SELECT * FROM public.dim_time", engine)
df_keyword  = pd.read_sql("SELECT * FROM public.dim_keyword", engine)
df_metrics  = pd.read_sql("SELECT * FROM public.fact_video_metrics", engine)
df_comments = pd.read_sql("SELECT * FROM public.fact_comments", engine)

print("dim_video:    ", df_video.shape)
print("dim_channel:  ", df_channel.shape)
print("dim_topic:    ", df_topic.shape)
print("dim_time:     ", df_time.shape)
print("dim_keyword:  ", df_keyword.shape)
print("fact_metrics: ", df_metrics.shape)
print("fact_comments:", df_comments.shape)

dim_video:     (20743, 10)
dim_channel:   (17729, 6)
dim_topic:     (10, 2)
dim_time:      (11317, 6)
dim_keyword:   (100, 3)
fact_metrics:  (74983, 7)
fact_comments: (1361046, 5)


In [18]:
df_keyword_clean = df_keyword.drop(columns=["topic_id"])

df_video_full = df_metrics \
    .merge(df_video,         on="video_id",   how="left") \
    .merge(df_channel,       on="channel_id", how="left") \
    .merge(df_topic,         on="topic_id",   how="left") \
    .merge(df_keyword_clean, on="keyword_id", how="left") \
    .merge(df_time,          on="time_id",    how="left")

print("Shape:", df_video_full.shape)
df_video_full.head()

Shape: (74983, 28)


,video_id,time_id,views,likes,comment_count,crawl_timestamp,is_trending,title,channel_id,thumbnail,...,subscriber_count,video_count,total_views,topic_name,keyword,year,month,day,hour,weekday
0,JJXKkfUYM9c,2026031012,95250,2781,2272,2026-03-10 12:40:34.402602,None,Honor X9d Review - Masakit man pero Kailangan !,UC7iCSSpCsNw34E1o0DwgJBA,https://i.ytimg.com/vi/JJXKkfUYM9c/default.jpg,...,156000.0,1210.0,3.478252e+07,technology,tech review,2026,3,10,12,1
1,JXZCpQsY0e4,2026031012,54979,4922,1660,2026-03-10 12:40:34.402602,None,"ReviewTechUSA Is Back (I Made A Mistake, Here&...",UCOuKecIw7ObKsLCMimUbbMA,https://i.ytimg.com/vi/JXZCpQsY0e4/default.jpg,...,92600.0,1298.0,2.017776e+07,technology,tech review,2026,3,10,12,1
2,mkcgVN90zl0,2026031012,11565022,556461,5344,2026-03-10 12:40:34.402602,None,This Gadget BRICKS Your Phone!,UCMiJRAwDNSNzuYeN2uWa0pA,https://i.ytimg.com/vi/mkcgVN90zl0/default.jpg,...,22300000.0,1886.0,8.356295e+09,technology,tech review,2026,3,10,12,1
3,ObqHKHoSE5Y,2026031012,4042883,123695,9221,2026-03-10 12:40:34.402602,None,Samsung S26 Ultra Hands on - What&#39;s ACTUAL...,UCMiJRAwDNSNzuYeN2uWa0pA,https://i.ytimg.com/vi/ObqHKHoSE5Y/default.jpg,...,22300000.0,1886.0,8.356295e+09,technology,tech review,2026,3,10,12,1
4,RvP-uVNwnXo,2026031012,3844230,90816,2448,2026-03-10 12:40:34.402602,None,Dope Tech: STILL so good!,UCBJycsmduvYEL83R_U4JriQ,https://i.ytimg.com/vi/RvP-uVNwnXo/default.jpg,...,20800000.0,1805.0,5.299716e+09,technology,tech review,2026,3,10,12,1


In [19]:
df_comments_full = df_comments \
    .merge(df_video[["video_id", "title", "topic_id"]], on="video_id", how="left") \
    .merge(df_topic, on="topic_id", how="left")

print("Comments dataset shape:", df_comments_full.shape)
df_comments_full.head()

Comments dataset shape: (1361046, 8)


,comment_id,video_id,time_id,comment_text,like_count,title,topic_id,topic_name
0,UgxIPKOQXWACvLf4fYF4AaABAg,JJXKkfUYM9c,2026012413,Sana ako mabunot need ko tlga cp kasi nag haha...,10,Honor X9d Review - Masakit man pero Kailangan !,7.0,technology
1,UgwOuKkXxF_M4l_PvZB4AaABAg,JJXKkfUYM9c,2026031004,Yan nlang bilhin ko❤🥰,0,Honor X9d Review - Masakit man pero Kailangan !,7.0,technology
2,UgzqCZ7H73rJ0rAAOAt4AaABAg,JJXKkfUYM9c,2026030909,Pang regalo lng lods sa anak ko salamat❤❤❤,0,Honor X9d Review - Masakit man pero Kailangan !,7.0,technology
3,UgysoXXsEaAiveRJ57R4AaABAg,JJXKkfUYM9c,2026030806,😂😂😂watching gamit honor x9d matibay nga po yan,0,Honor X9d Review - Masakit man pero Kailangan !,7.0,technology
4,UgxqqhHLeLNn_bYMqBJ4AaABAg,JJXKkfUYM9c,2026030800,Honox9d din unit ko pero dko susubukan yan😅😅😅😅,0,Honor X9d Review - Masakit man pero Kailangan !,7.0,technology


In [20]:
df_video_full.to_csv("../data_merge/df_video_full.csv", index=False)
df_comments_full.to_csv("../data_merge/df_comments_full.csv", index=False)

print("Xuất xong!")
print("df_video_full:    ", df_video_full.shape)
print("df_comments_full: ", df_comments_full.shape)

Xuất xong!
df_video_full:     (74983, 28)
df_comments_full:  (1361046, 8)
